# Project 16 - Smart Meters London - Exploratory Data Analysis

**Dataset:** Smart Meters in London (Kaggle: jeanmidev/smart-meters-in-london)
**Window:** Nov 2011 - Feb 2014
**Households:** 5,567 (UK Power Networks Low Carbon London trial)
**Granularity:** Half-hourly kWh per household, plus hourly weather and ACORN demographic groups.

This notebook is the raw scaffold for EDA. Cells are not pre-executed; the main session runs them after the dataset has been pulled into `../data/`.

## Plan

1. Load metadata files (`informations_households.csv`, `acorn_details.csv`, `uk_bank_holidays.csv`).
2. Inspect a single half-hourly block to understand schema, dtypes, and missingness.
3. Aggregate one block to daily totals and check seasonality (weekly + yearly).
4. Cross-reference weather (`weather_hourly_darksky.csv`).
5. Visualise per-household load profiles vs ACORN group.
6. Identify cold-start households (less than 3 months of history).
7. Flag the dToU (dynamic Time-of-Use) cohort vs Std (flat) cohort.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_DIR = Path('../data')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 220)
sns.set_style('whitegrid')

## 1. Households metadata

In [ ]:
households = pd.read_csv(DATA_DIR / 'informations_households.csv')
print('Shape:', households.shape)
households.head()

In [ ]:
households['stdorToU'].value_counts(dropna=False)

In [ ]:
households['Acorn_grouped'].value_counts(dropna=False)

In [ ]:
households['Acorn'].value_counts(dropna=False).head(20)

## 2. Half-hourly block schema and missingness

We inspect `block_0` first; the full daily corpus is read later in chunked form for the baseline.

In [ ]:
block0 = pd.read_csv(
    DATA_DIR / 'halfhourly_dataset' / 'halfhourly_dataset' / 'block_0.csv',
    parse_dates=['tstp'],
    na_values=['Null'],
)
block0['energy(kWh/hh)'] = pd.to_numeric(block0['energy(kWh/hh)'], errors='coerce')
print('Shape:', block0.shape)
block0.head()

In [ ]:
block0.info()

In [ ]:
block0.describe()

In [ ]:
missingness = block0.isna().mean().sort_values(ascending=False)
missingness

In [ ]:
households_in_block0 = block0['LCLid'].nunique()
print(f'Distinct households in block_0: {households_in_block0}')
print('Date range:', block0['tstp'].min(), 'to', block0['tstp'].max())

## 3. Target distribution

Half-hourly kWh is right-skewed and bounded below by zero. We expect most readings under 0.5 kWh per half-hour, with a long tail driven by EV charging, electric showers, and resistive heating spikes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
block0['energy(kWh/hh)'].hist(bins=80, ax=axes[0])
axes[0].set_title('Half-hourly kWh - linear scale')
axes[0].set_xlabel('kWh per half-hour')
block0['energy(kWh/hh)'].clip(lower=1e-4).apply(np.log10).hist(bins=80, ax=axes[1])
axes[1].set_title('Half-hourly kWh - log10 scale')
axes[1].set_xlabel('log10(kWh per half-hour)')
plt.tight_layout()
plt.show()

In [ ]:
# Per-household summary
per_hh = block0.groupby('LCLid')['energy(kWh/hh)'].agg(['count', 'mean', 'std', 'min', 'max'])
per_hh.describe()

## 4. Daily aggregates and weekly seasonality

Aggregate one household to daily kWh and look at the weekday vs weekend split. We expect higher daytime weekend consumption and a clear morning + evening shoulder on weekdays.

In [ ]:
sample_id = block0['LCLid'].value_counts().index[0]
sample = block0[block0['LCLid'] == sample_id].copy()
sample = sample.set_index('tstp').sort_index()
daily = sample['energy(kWh/hh)'].resample('1D').sum()
daily.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
daily.plot(ax=ax)
ax.set_title(f'Daily total kWh - household {sample_id}')
ax.set_ylabel('kWh / day')
plt.show()

In [ ]:
# Weekly profile: average half-hourly kWh by weekday and half-hour-of-day
sample['weekday'] = sample.index.dayofweek
sample['hh_of_day'] = sample.index.hour * 2 + (sample.index.minute // 30)
profile = sample.groupby(['weekday', 'hh_of_day'])['energy(kWh/hh)'].mean().unstack(0)
fig, ax = plt.subplots(figsize=(10, 4))
profile.plot(ax=ax)
ax.set_title(f'Half-hourly kWh by weekday - household {sample_id}')
ax.set_xlabel('Half-hour of day (0 = midnight, 47 = 11:30 pm)')
ax.set_ylabel('Mean kWh')
ax.legend(title='Weekday (0 = Mon)', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 5. Weather covariates

Hourly Darksky weather is bundled. We will use temperature, humidity, wind speed, and cloud cover as exogenous covariates in the SARIMA baseline and as known-future covariates in the TFT advanced model.

In [ ]:
weather = pd.read_csv(DATA_DIR / 'weather_hourly_darksky.csv', parse_dates=['time'])
print('Shape:', weather.shape)
weather.head()

In [ ]:
weather[['temperature', 'humidity', 'windSpeed', 'cloudCover']].describe()

## 6. ACORN-level aggregates

Hierarchical structure: household -> ACORN class -> ACORN supergroup -> all-trial total. The advanced model groups by ACORN supergroup (~6 levels) for the TFT static categorical input. The reconciliation discussion uses MinT (Wickramasuriya 2019).

In [ ]:
acorn_details = pd.read_csv(DATA_DIR / 'acorn_details.csv')
print('Shape:', acorn_details.shape)
acorn_details.head()

## 7. Cold-start and dToU flagging

- Cold-start: households with fewer than 90 distinct days in the trial. Drop from baseline; keep in advanced with a `short_history` static categorical feature.
- dToU cohort: a subset on dynamic Time-of-Use tariff (six high-price events per year, signalled day-ahead). We treat them as a separate static-feature group in the TFT model and discuss treatment-effect estimation in Discussion.

In [ ]:
history_per_hh = (
    block0.assign(date=block0['tstp'].dt.date)
    .groupby('LCLid')['date']
    .nunique()
    .sort_values()
)
cold_start = (history_per_hh < 90).sum()
print(f'Cold-start households in block_0: {cold_start} / {len(history_per_hh)}')
history_per_hh.describe()

## 8. Next steps

1. Concatenate all blocks lazily via `pyarrow` and write a parquet partitioned by `LCLid` for efficient per-household reads.
2. Resample to a regular 30-minute grid; mark gap-filled half-hours with a missingness mask.
3. Run `src/model_baseline.py` for SARIMA / Prophet on a sample of 50 households (sanity check), then full sweep.
4. Run `src/model_advanced.py` for the TFT model; train on 80% of households, hold out 20% for cold-start generalisation.